In [51]:
from llama_parse import LlamaParse
from dotenv import load_dotenv
load_dotenv()
import os
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LLAMA_CLOUD_API_KEY"] = os.getenv("LLAMA_API_KEY")
def config_parser(pdf_path):
    # 1. Parse PDF sang Markdown
    parser = LlamaParse(
        result_type="markdown",
        auto_mode=True,
        auto_mode_trigger_on_image_in_page=True,
        auto_mode_trigger_on_table_in_page=True,
        skip_diagonal_text=True,
        preserve_layout_alignment_across_pages=True,
        num_workers=4,
        max_timeout=1000,
        preserve_very_small_text=True,
    )
    file_name = os.path.splitext(os.path.basename(pdf_path))[0]
    print("Đang parse PDF sang Markdown...")
    parsed_docs = parser.load_data(pdf_path)  # Mỗi trang PDF -> 1 Document dạng markdown
    return parsed_docs, file_name

In [52]:
parsed_docs, file_name = config_parser("D:/project/technical_requirements/downloads/DC Power Systems/ESURE™ RECTIFIER 1000e3/R48-1000e3 User Manual.pdf")

Đang parse PDF sang Markdown...
Started parsing the file under job_id 8e92e116-c936-43c7-bd63-1bef0fe2fe33
.

In [53]:
parsed_docs

[Document(id_='9f213ea6-a0ed-43d6-b5d4-eb31e4053f24', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='\n\nVertiv logo\n\n# NetSure™\n\n## Rectifier Module\n\nUser Manual (UM1R48100003 / 11MB4705YO), Revision B\n\nSpecification Number: 1R48100003\nModel Number: R48-100003\n\n[Line drawing of a rectifier module showing a rectangular unit with a fan at one end and mounting holes]\n', path=None, url=None, mimetype=None), image_resource=None, audio_resource=None, video_resource=None, text_template='{metadata_str}\n\n{content}'),
 Document(id_='c8573585-8f50-40e5-9912-a2f56860359c', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, te

In [54]:
content_pages = []

In [55]:
from llama_index.core import Document
import re
for item in parsed_docs:
    text = item.text
    cleaned = re.sub(r"```", "", text)
    cleaned = re.sub("markdown", "", cleaned)
    content_pages.append(Document(text = cleaned))

In [56]:
content_pages

[Document(id_='59078e43-f4fc-4eff-941f-f9ca3bf2ec4f', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='\n\nVertiv logo\n\n# NetSure™\n\n## Rectifier Module\n\nUser Manual (UM1R48100003 / 11MB4705YO), Revision B\n\nSpecification Number: 1R48100003\nModel Number: R48-100003\n\n[Line drawing of a rectifier module showing a rectangular unit with a fan at one end and mounting holes]\n', path=None, url=None, mimetype=None), image_resource=None, audio_resource=None, video_resource=None, text_template='{metadata_str}\n\n{content}'),
 Document(id_='9fbc9bd0-ea1e-4b48-b6b1-734cd6cc0a9c', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, te

In [57]:
from llama_index.core import Document
from llama_index.core.node_parser import MarkdownNodeParser

md_parser = MarkdownNodeParser(include_metadata=True, include_prev_next_rel=True)
nodes = md_parser.get_nodes_from_documents(content_pages)

In [58]:
len(nodes)

75

In [59]:
output_dir = "D:/project/technical_requirements/output_test"
os.makedirs(output_dir, exist_ok=True)
output_file = f"{output_dir}/{file_name}.md"
with open(output_file, "w", encoding="utf-8") as f:
    for idx, node in enumerate(nodes, start=1):
        f.write(f"# Chunk {idx}\n")
        f.write(node.text + "\n\n\n\n")

print(f"Đã ghi {len(nodes)} chunk vào {output_file}")

Đã ghi 75 chunk vào D:/project/technical_requirements/output_test/R48-1000e3 User Manual.md
